In [71]:
import pandas as pd
import numpy as np
import textwrap
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk import word_tokenize, sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [72]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\gupta\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gupta\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## TextRank Text Summarization

In [73]:
df = pd.read_csv('BBC_Text_CLS.csv')
df.head(5)

,text,labels
0,Ad sales boost Time Warner profit\n\nQuarterly...,business
1,Dollar gains on Greenspan speech\n\nThe dollar...,business
2,Yukos unit buyer faces loan claim\n\nThe owner...,business
3,High fuel prices hit BA's profits\n\nBritish A...,business
4,Pernod takeover talk lifts Domecq\n\nShares in...,business


In [74]:
df['labels'].value_counts()

labels
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64

In [75]:
doc = df[df.labels == 'business']['text'].sample(random_state=42)

In [76]:
def wrap(x):
    return textwrap.fill(x, replace_whitespace=False, fix_sentence_endings=True)

In [77]:
print(wrap(doc.iloc[0]))

Christmas sales worst since 1981

UK retail sales fell in December,
failing to meet expectations and making it by some counts the worst
Christmas since 1981.

Retail sales dropped by 1% on the month in
December, after a 0.6% rise in November, the Office for National
Statistics (ONS) said.  The ONS revised the annual 2004 rate of growth
down from the 5.9% estimated in November to 3.2%. A number of
retailers have already reported poor figures for December.  Clothing
retailers and non-specialist stores were the worst hit with only
internet retailers showing any significant growth, according to the
ONS.

The last time retailers endured a tougher Christmas was 23 years
previously, when sales plunged 1.7%.

The ONS echoed an earlier
caution from Bank of England governor Mervyn King not to read too much
into the poor December figures.  Some analysts put a positive gloss on
the figures, pointing out that the non-seasonally-adjusted figures
showed a performance comparable with 2003. The Novembe

In [78]:
sents = nltk.sent_tokenize(doc.iloc[0].split('\n',1)[1])
sents

['\nUK retail sales fell in December, failing to meet expectations and making it by some counts the worst Christmas since 1981.',
 'Retail sales dropped by 1% on the month in December, after a 0.6% rise in November, the Office for National Statistics (ONS) said.',
 'The ONS revised the annual 2004 rate of growth down from the 5.9% estimated in November to 3.2%.',
 'A number of retailers have already reported poor figures for December.',
 'Clothing retailers and non-specialist stores were the worst hit with only internet retailers showing any significant growth, according to the ONS.',
 'The last time retailers endured a tougher Christmas was 23 years previously, when sales plunged 1.7%.',
 'The ONS echoed an earlier caution from Bank of England governor Mervyn King not to read too much into the poor December figures.',
 'Some analysts put a positive gloss on the figures, pointing out that the non-seasonally-adjusted figures showed a performance comparable with 2003.',
 'The November-De

In [79]:
len(sents)  # Total no of sentences

17

In [80]:
featurizer = TfidfVectorizer(stop_words=stopwords.words('english'),
                             norm='l1')

In [81]:
X = featurizer.fit_transform(sents)

X.shape

(17, 154)

In [82]:
S = cosine_similarity(X)  # Similarity matrix of sentences

S.shape

(17, 17)

In [83]:
# Normalize similarity matrix
S = S / S.sum(axis=1, keepdims=True)

In [84]:
S[0].sum()  # Check if the sum of each row is 1

np.float64(1.0)

In [85]:
# Uniform Transition Matrix
U = np.ones_like(S) / len(S)

U.shape

(17, 17)

In [86]:
U[0].sum()  # Check if the sum of each row is 1

np.float64(1.0)

In [87]:
# Smoothed Similarity Matrix
factor = 0.15

S = (1-factor)* S + factor * U

In [88]:
eigenvals , eigenvecs = np.linalg.eig(S.transpose())

In [89]:
eigenvals

array([1.        , 0.24245466, 0.72108199, 0.67644122, 0.34790129,
       0.34417302, 0.3866884 , 0.40333562, 0.41608572, 0.44238593,
       0.63909999, 0.62556792, 0.58922572, 0.57452382, 0.48511399,
       0.51329157, 0.52975372])

In [90]:
eigenvecs[:, 0]

array([-0.24206557, -0.27051337, -0.2213806 , -0.28613638, -0.25065894,
       -0.2499217 , -0.279622  , -0.21515455, -0.2226665 , -0.22745415,
       -0.2059112 , -0.20959727, -0.23526242, -0.24203809, -0.23663025,
       -0.2940483 , -0.20865607])

In [91]:
eigenvecs[:,0].dot(S)

array([-0.24206557, -0.27051337, -0.2213806 , -0.28613638, -0.25065894,
       -0.2499217 , -0.279622  , -0.21515455, -0.2226665 , -0.22745415,
       -0.2059112 , -0.20959727, -0.23526242, -0.24203809, -0.23663025,
       -0.2940483 , -0.20865607])

In [92]:
limiting_dist = np.ones(len(S)) / len(S)
threshold = 1e-8
delta = float('inf')

iters = 0
while delta > threshold:

    iters +=1

    p = limiting_dist.dot(S) # Markov Transition

    delta = np.abs(p - limiting_dist).sum() # Difference between the two distributions

    limiting_dist = p  # Update  limiting distribution

In [93]:
print(iters)

41


In [94]:
limiting_dist

array([0.05907327, 0.06601563, 0.05402534, 0.06982824, 0.06117038,
       0.06099047, 0.06823848, 0.05250595, 0.05433915, 0.05550753,
       0.05025022, 0.05114977, 0.05741304, 0.05906657, 0.05774685,
       0.07175905, 0.05092008])

In [95]:
scores = limiting_dist

In [96]:
sort_idx = np.argsort(-scores)

In [97]:
print("Generated Summary:\n")

for i in sort_idx[:5]: # Print top 5 sentences
    print(wrap("%.2f" % scores[i] + " -- " + sents[i]))

Generated Summary:

0.07 -- "The retail sales figures are very weak, but as Bank of
England governor Mervyn King indicated last night, you don't really
get an accurate impression of Christmas trading until about Easter,"
said Mr Shaw.
0.07 -- A number of retailers have already reported poor figures for
December.
0.07 -- The ONS echoed an earlier caution from Bank of England
governor Mervyn King not to read too much into the poor December
figures.
0.07 -- Retail sales dropped by 1% on the month in December, after a
0.6% rise in November, the Office for National Statistics (ONS) said.
0.06 -- Clothing retailers and non-specialist stores were the worst
hit with only internet retailers showing any significant growth,
according to the ONS.


In [98]:
doc.iloc[0].split('\n',1)[0]   # Title of the document

'Christmas sales worst since 1981'

In [99]:
def summarize(text):

    sents = sent_tokenize(text) # Extract sentences from text
    X = featurizer.fit_transform(sents) # Perform TF-IDF vectorization

    S = cosine_similarity(X) # Compute similarity matrix
    S = S / S.sum(axis=1, keepdims=True) # Normalize similarity matrix

    U = np.ones_like(S) / len(S)   # Uniform Transition Matrix
    S = (1-factor)* S + factor * U   # Smoothed Similarity Matrix

    eigenvals , eigenvecs = np.linalg.eig(S.transpose()) # Eigen decomposition

    scores = eigenvecs[:,0] / eigenvecs[:,0].sum() # Compute scores

    sort_idx = np.argsort(-scores) # Sort sentences based on scores

    print("Generated Summary:\n")
    for i in sort_idx[:5]: # Print top 5 sentences
        print(wrap("%.2f" % scores[i] + " -- " + sents[i]))

In [103]:
doc = df[df.labels == 'entertainment']['text'].sample()


In [104]:
doc.iloc[0].split('\n',1)[0]  # Title of the article

'New Harry Potter tops book chart'

In [105]:
summarize(doc.iloc[0].split('\n',1)[1])

Generated Summary:

0.09 -- Rowling has already revealed that the Half-Blood Prince is
neither Harry nor Voldemort.
0.09 -- Customers who pre-order Harry Potter and the Half-Blood Prince
avoid standing in long queues at bookshops on the day of the book's
release next July.
0.09 -- 
Harry Potter and the Half-Blood Prince has topped Amazon's
book chart less than 24 hours after its release date - 16 July - was
announced.
0.08 -- Thousands of customers placed pre-orders on the amazon.co.uk
website for the sixth book in the series.
0.08 -- "The fact that the book has already hit number one in our Hot
100 books chart shows how incredibly excited customers are about the
sixth Harry Potter," said Amazon's Robin Terrell.


## Libraries for Text Summarization

In [106]:
from sumy.summarizers.text_rank import TextRankSummarizer
from sumy.summarizers.lsa import LsaSummarizer

from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer

In [107]:
summarizer = TextRankSummarizer()

parser = PlaintextParser.from_string(doc.iloc[0].split('\n',1)[1], 
                                     Tokenizer("english"))

In [ ]:
summary = summarizer(parser.document, sentences_count=5)  # Summarize the document with 5 sentences

In [110]:
for s in summary:
    print(wrap(str(s)))  # Print the summary sentences

Thousands of customers placed pre-orders on the amazon.co.uk website
for the sixth book in the series.
JK Rowling's fifth book in the wizard series, Harry Potter and Order
of the Phoenix was Amazon's largest pre-ordered item ever, with
420,000 copies pre-ordered prior to its release in June 2003.
Customers who pre-order Harry Potter and the Half-Blood Prince avoid
standing in long queues at bookshops on the day of the book's release
next July.
"The fact that the book has already hit number one in our Hot 100
books chart shows how incredibly excited customers are about the sixth
Harry Potter," said Amazon's Robin Terrell.
Harry Potter and the Half-Blood Prince takes up the story of Harry
Potter's sixth year at Hogwarts School of Witchcraft and Wizardry as
Lord Voldemort grows stronger.


In [111]:
summarizer = LsaSummarizer()

summary = summarizer(parser.document, sentences_count=5)  # Summarize the document with 5 sentences

In [112]:
for s in summary:
    print(wrap(str(s)))  # Print the summary sentences

Harry Potter and the Half-Blood Prince has topped Amazon's book chart
less than 24 hours after its release date - 16 July - was announced.
JK Rowling's fifth book in the wizard series, Harry Potter and Order
of the Phoenix was Amazon's largest pre-ordered item ever, with
420,000 copies pre-ordered prior to its release in June 2003.
Harry Potter and the Half-Blood Prince takes up the story of Harry
Potter's sixth year at Hogwarts School of Witchcraft and Wizardry as
Lord Voldemort grows stronger.
Rowling has already revealed that the Half-Blood Prince is neither
Harry nor Voldemort.
Rowling said she had plenty of time while pregnant "to tinker with the
manuscript to my satisfaction and I am as happy as I have ever been
with the end result".


In [115]:
from summa.summarizer import summarize

In [120]:
summary = summarize(doc.iloc[0].split('\n',1)[1], 
                    words=100)  # Summarize the document with 100 words

In [121]:
print(wrap(summary))  # Print the summary

Harry Potter and the Half-Blood Prince has topped Amazon's book chart
less than 24 hours after its release date - 16 July - was announced.
Thousands of customers placed pre-orders on the amazon.co.uk website
for the sixth book in the series.
JK Rowling's fifth book in the
wizard series, Harry Potter and Order of the Phoenix was Amazon's
largest pre-ordered item ever, with 420,000 copies pre-ordered prior
to its release in June 2003.
Customers who pre-order Harry Potter and
the Half-Blood Prince avoid standing in long queues at bookshops on
the day of the book's release next July.
